In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ========================
# PARTE 0: INSTALAÇÕES
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]

def buscar_links_por_categoria(termos, max_paginas=50):
    links = set()
    for termo in termos:
        print(f"🔍 Buscando: {termo}")
        for pagina in range(max_paginas):
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=100" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200: break
            data = res.json()
            hits = data.get("hits", [])
            if not hits: break
            for item in hits:
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    links.add(full_url)
            if pagina >= data.get("nbPages", 0) - 1:
                break
    return links

print("🔍 Coletando links...")
todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO PRODUTOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    seletores = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer',
        '.price',
        '.price__value'
    ]
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break
    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETA E PLANILHAS NO DRIVE
# ========================
todos = []
for i, link in enumerate(sorted(todos_links), 1):
    print(f"🔗 {i}/{len(todos_links)} -> {link}")
    try:
        dados = extrair_dados_produto(link)
        todos.append(dados)
    except Exception as e:
        print("Erro:", e)
driver.quit()

# Salvar em: Meu Drive/WebScraping/FEITO/Leroy/COLETA_LEROY
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY"
os.makedirs(base_drive, exist_ok=True)

arquivo_excel = os.path.join(base_drive, f"leroymerelin_{data_nome}.xlsx")
arquivo_csv   = os.path.join(base_drive, f"leroymerelin_{data_nome}.csv")

df = pd.DataFrame(todos)
df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df = df[colunas]
df.to_excel(arquivo_excel, index=False)
df.to_csv(arquivo_csv, index=False, sep=";")

print(f"✅ Excel salvo em: {arquivo_excel}")
print(f"✅ SCV salvo em: {arquivo_csv}")

# ========================
# PARTE 5: DOWNLOAD DE IMAGENS NO DRIVE
# ========================
pasta_imagens = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}")
os.makedirs(pasta_imagens, exist_ok=True)

for _, r in df.iterrows():
    url_img = r['Imagem']
    cod = r['Código']
    nome_arquivo = f"{cod}.png"
    caminho = os.path.join(pasta_imagens, nome_arquivo)

    if url_img and url_img != "N/A" and not os.path.exists(caminho):
        for _ in range(3):
            try:
                resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(caminho, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
                    print(f"✅ Baixada: {nome_arquivo}")
                    break
            except Exception as e:
                print(f"⚠️ Erro para {cod}: {e}")
                time.sleep(2)
    else:
        print(f"⏩ Já existe: {nome_arquivo}")

# Zipar imagens
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"📦 Imagens zipadas em: {zipname}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Coletando links...
🔍 Buscando: pisos
🔍 Buscando: revestimentos
🔍 Buscando: porcelanatos
✅ Total de links coletados: 2527

🔗 1/2527 -> https://www.leroymerlin.com.br/mosaico-pedra-lm-0916-30x30cm-anticatto_89741211
🔗 2/2527 -> https://www.leroymerlin.com.br/piso-ceramico-acetinado-20x20cm-caixa-1,18m2-incepa-tribal-cimento-62030024_1572286431


KeyboardInterrupt: 

In [ ]:
# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile
from datetime import datetime

# Configurar Selenium
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]

def buscar_links_por_categoria(termos, max_paginas=50):
    links = set()
    for termo in termos:
        print(f"🔍 Buscando: {termo}")
        for pagina in range(max_paginas):
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=100" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200: break
            data = res.json()
            hits = data.get("hits", [])
            if not hits: break
            for item in hits:
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    links.add(full_url)
            if pagina >= data.get("nbPages", 0) - 1:
                break
    return links

print("🔍 Coletando links...")
todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE PRODUTOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    seletores = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer',
        '.price',
        '.price__value'
    ]
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break
    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETA DE DADOS E IMAGENS
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

todos = []
for i, link in enumerate(sorted(todos_links), 1):
    try:
        print(f"🔗 {i}/{len(todos_links)} -> {link}")
        dados = extrair_dados_produto(link)
        todos.append(dados)

        # Baixar imagem se ainda não existir
        codigo = dados.get("Código", "sem_codigo")
        imagem_path = os.path.join(pasta_imagens, f"{codigo}.png")
        url_img = dados.get("Imagem")

        if not os.path.exists(imagem_path) and url_img and url_img != "N/A":
            try:
                resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(imagem_path, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
                    print(f"✅ Imagem baixada: {codigo}.png")
                else:
                    print(f"⚠️ Imagem não encontrada para {codigo}")
            except Exception as e:
                print(f"⚠️ Erro ao baixar imagem de {codigo}: {e}")
        else:
            print(f"⏩ Imagem já existe: {codigo}.png")

    except Exception as e:
        print("❌ Erro ao processar:", e)

driver.quit()

# ========================
# PARTE 5: SALVAR PLANILHAS E ZIPAR IMAGENS
# ========================
arquivo_excel = os.path.join(base_drive, f"leroymerelin_{data_nome}.xlsx")
arquivo_csv   = os.path.join(base_drive, f"leroymerelin_{data_nome}.csv")

df = pd.DataFrame(todos)
df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df = df[colunas]
df.to_excel(arquivo_excel, index=False)
df.to_csv(arquivo_csv, index=False, sep=";")

# Zipar imagens
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"\n✅ Excel salvo em: {arquivo_excel}")
print(f"✅ CSV salvo em: {arquivo_csv}")
print(f"📦 Imagens zipadas em: {zipname}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Coletando links...
🔍 Buscando: pisos
🔍 Buscando: revestimentos
🔍 Buscando: porcelanatos
✅ Total de links coletados: 2527

🔗 1/2527 -> https://www.leroymerlin.com.br/mosaico-pedra-lm-0916-30x30cm-anticatto_89741211
✅ Imagem baixada: 89741211.png
🔗 2/2527 -> https://www.leroymerlin.com.br/piso-ceramico-acetinado-20x20cm-caixa-1,18m2-incepa-tribal-cimento-62030024_1572286431
✅ Imagem baixada: 1572286431.png
🔗 3/2527 -> https://www.leroymerlin.com.br/piso-ceramico-acetinado-20x23cm-caixa-1,13m2-incepa-seattle-gris-bold-inc07490001a_1572286452
✅ Imagem baixada: 1572286452.png
🔗 4/2527 